# tests.ipynb — single source of truth (writes reward.json)

> **IMPORTANT — Colab runtime.** Before running:
>   1. `Runtime → Change runtime type → Hardware accelerator: CPU`
>   2. Runtime version: any (Latest is fine; this notebook works on
>      Python 3.10, 3.11, and 3.12 because scikit-image v0.22.0
>      ships binary wheels for all of them).
>
> Then `Runtime → Run all`. Whole notebook is ~10–15 minutes.


## What this notebook does
1. Install scikit-image v0.22.0 (baseline).
2. Build workload, run `regionprops_table` -> golden output,
   run existing tests, benchmark baseline.
3. Apply our fast-path patch.
4. Run existing tests on patched install. Any baseline-pass
   that now fails = regression.
5. Re-run workload, compare candidate vs golden with
   `np.allclose(rtol=1e-6, atol=1e-6)`.
6. Benchmark candidate.
7. Compute speedup, write `reward.json`.


## 1. Runtime info

In [ ]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version


## 2. Clone the submission repo

In [ ]:
# ====================================================================
# CONFIG  (usually nothing to change)
# --------------------------------------------------------------------
# SUBMISSION_REPO_URL = the public repo holding the patch, workload
# generator, and benchmark utils. Override via env var if forking.
# ====================================================================
import os
SUBMISSION_REPO_URL = os.environ.get(
    "SUBMISSION_REPO_URL",
    "https://github.com/Sanjay040695/scikit-image-performance-optimization.git"
)
SUBMISSION_DIR = "/content/submission"
print("Using submission repo:", SUBMISSION_REPO_URL)

# Always re-clone so we pick up any push you made after a previous run.
!rm -rf $SUBMISSION_DIR
!git clone --depth 1 $SUBMISSION_REPO_URL $SUBMISSION_DIR

import sys
for p in (SUBMISSION_DIR,
          os.path.join(SUBMISSION_DIR, "workloads"),
          os.path.join(SUBMISSION_DIR, "benchmark"),
          os.path.join(SUBMISSION_DIR, "patches")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("OK — submission repo at:", SUBMISSION_DIR)


## 3. Install scikit-image v0.22.0

In [ ]:
# ====================================================================
# Install scikit-image v0.24.0 (June 2024 — ~23 months old, comfortably
# >12 months per the brief). Its wheel is numpy-2.x ABI compatible, so
# we do NOT touch numpy/scipy — those are kept at Colab's defaults,
# which avoids the in-memory-vs-on-disk mismatches that earlier
# numpy upgrades created.
# ====================================================================
import sys, subprocess, os

# Install (or downgrade) scikit-image to exactly v0.24.0. Nothing else.
!pip install -q --upgrade --no-deps scikit-image==0.24.0 2>&1 | tail -3

# Force-reload so any previously imported skimage is dropped.
for k in list(sys.modules):
    if k.startswith("skimage"):
        del sys.modules[k]

import numpy, scipy, skimage
print(f"numpy:   {numpy.__version__}")
print(f"scipy:   {scipy.__version__}")
print(f"skimage: {skimage.__version__}    (expected 0.24.0)")
assert skimage.__version__ == "0.24.0", (
    f"Expected scikit-image 0.24.0 but got {skimage.__version__}. "
    "Install did not take effect."
)

# Record the SHA so reward.json can reference it.
BASELINE_SHA = subprocess.run(
    ["git", "ls-remote", "https://github.com/scikit-image/scikit-image.git", "refs/tags/v0.24.0"],
    capture_output=True, text=True,
).stdout.split()[0]
print(f"baseline tag v0.24.0 resolves to SHA: {BASELINE_SHA}")
with open("/content/baseline_sha.txt", "w") as f:
    f.write(BASELINE_SHA)

print()
print("OK — environment is ready.")


## 4. Workload + baseline measurements

In [ ]:
from generate_workload import build_workload, WORKLOAD_PROPERTIES
from benchmark_utils import bench
import subprocess, json, os, numpy as np, sys, skimage

labels, intensity, n_regions = build_workload()
print(f"workload: shape={labels.shape}, n_regions={n_regions}")

from skimage.measure import regionprops_table
golden = regionprops_table(labels, intensity_image=intensity,
                           properties=list(WORKLOAD_PROPERTIES))
os.makedirs("/content/outputs", exist_ok=True)
np.savez_compressed("/content/outputs/golden_output.npz", **golden)

test_dir = os.path.join(os.path.dirname(skimage.__file__),
                        "measure", "tests")
proc = subprocess.run(
    ["pytest", os.path.join(test_dir, "test_regionprops.py"),
     "-v", "--tb=no", "-q", "--no-header"],
    capture_output=True, text=True,
)
baseline_passed, baseline_failed = [], []
for line in proc.stdout.splitlines():
    if " PASSED" in line:
        baseline_passed.append(line.split(" PASSED")[0].strip())
    elif " FAILED" in line:
        baseline_failed.append(line.split(" FAILED")[0].strip())
print(f"baseline: {len(baseline_passed)} pass, {len(baseline_failed)} fail")
with open("/content/outputs/baseline_tests.json", "w") as f:
    json.dump({"passed": sorted(baseline_passed),
               "failed": sorted(baseline_failed)}, f, indent=2)

baseline_stats = bench(
    lambda: regionprops_table(labels, intensity_image=intensity,
                              properties=list(WORKLOAD_PROPERTIES)),
    label="baseline", n_warmup=2, n_measured=7,
)


## 5. Apply patch + candidate measurements

In [ ]:
!python $SUBMISSION_DIR/patches/apply_optimization.py
for k in list(sys.modules):
    if k.startswith("skimage"):
        del sys.modules[k]
from skimage.measure import regionprops_table
from skimage.measure import _regionprops_fast  # raises if patch missing
print("patch active.")

proc = subprocess.run(
    ["pytest", os.path.join(test_dir, "test_regionprops.py"),
     "-v", "--tb=no", "-q", "--no-header"],
    capture_output=True, text=True,
)
cand_passed, cand_failed = [], []
for line in proc.stdout.splitlines():
    if " PASSED" in line:
        cand_passed.append(line.split(" PASSED")[0].strip())
    elif " FAILED" in line:
        cand_failed.append(line.split(" FAILED")[0].strip())
regressions = sorted(set(baseline_passed) - set(cand_passed))
print(f"candidate: {len(cand_passed)} pass, {len(cand_failed)} fail")
print(f"regressions: {len(regressions)}")
for r in regressions[:10]:
    print(" ", r)
existing_tests_pass = len(regressions) == 0

candidate_out = regionprops_table(labels, intensity_image=intensity,
                                  properties=list(WORKLOAD_PROPERTIES))
np.savez_compressed("/content/outputs/candidate_output.npz",
                    **candidate_out)
candidate_stats = bench(
    lambda: regionprops_table(labels, intensity_image=intensity,
                              properties=list(WORKLOAD_PROPERTIES)),
    label="candidate", n_warmup=2, n_measured=7,
)


## 6. Correctness + speedup + reward.json

In [ ]:
from benchmark_utils import compare_tables, write_json, collect_environment
ok, details = compare_tables(dict(golden), dict(candidate_out),
                             rtol=1e-6, atol=1e-6)
for k, v in details["columns"].items():
    print(f"  {k:30s} match={v['match']}  "
          f"max_abs_diff={v.get('max_abs_diff', 'n/a')}")
output_equivalent = ok
print(f"\noutputs equivalent within 1e-6: {output_equivalent}")

speedup = (baseline_stats["median"] / candidate_stats["median"]
           if existing_tests_pass and output_equivalent else None)

reward = {
    "repo": "scikit-image/scikit-image",
    "baseline_sha": BASELINE_SHA,
    "candidate_description": (
        "Vectorized regionprops_table fast path: scipy.ndimage.find_objects "
        "for bboxes, np.bincount for areas/centroid sums/weighted centroid "
        "sums, scipy.ndimage.minimum/maximum for per-label intensity extrema; "
        "replaces the per-region Python dispatch loop for the common scalar "
        "property set, with safe fallback to the original implementation."
    ),
    "baseline_time_s":  {k: baseline_stats[k]
                         for k in ("median", "iqr", "n_warmup", "n_measured")},
    "candidate_time_s": {k: candidate_stats[k]
                         for k in ("median", "iqr", "n_warmup", "n_measured")},
    "speedup": (float(speedup) if speedup is not None else None),
    "correctness": {
        "existing_tests_pass": bool(existing_tests_pass),
        "output_equivalent":   bool(output_equivalent),
        "tolerance": "1e-6 relative",
        "tolerance_justification": (
            "np.bincount accumulates sums across the full image; the baseline "
            "uses per-region np.sum. Both compute identical mathematical "
            "quantities but in different orders, so float results differ at "
            "~1e-12 relative. Integer columns must match exactly."
        ),
    },
    "environment": collect_environment(),
}
write_json("/content/reward.json", reward)
print(json.dumps(reward, indent=2))

print("=" * 60)
print(f"BASELINE  median = {baseline_stats['median']:.4f}s  "
      f"IQR = {baseline_stats['iqr']:.4f}s")
print(f"CANDIDATE median = {candidate_stats['median']:.4f}s  "
      f"IQR = {candidate_stats['iqr']:.4f}s")
if reward["speedup"] is not None:
    print(f"SPEEDUP = {reward['speedup']:.2f}x")
else:
    print("SPEEDUP = null  (correctness failed)")
print(f"existing_tests_pass = {reward['correctness']['existing_tests_pass']}")
print(f"output_equivalent   = {reward['correctness']['output_equivalent']}")
print("=" * 60)
print("reward.json written to /content/reward.json")
print("Download it from the Files panel (folder icon, left sidebar).")
